# SmartPantry AI | Role 5: Recipe Matching and Ranking


**SmartPantry AI | Recipe Recommendation and Safety Validation**

This notebook demonstrates SmartPantry AI's recipe matching and ranking algorithm using controlled sample data.

The original demonstration evaluates ingredient matching, pantry coverage, and expiration-based prioritization.

Additional automated tests validate Spoonacular response handling, recipe ranking integration, recall safety filtering, and verified recipe alternatives.

The notebook uses controlled data and mocked API responses to validate functionality without requiring paid API requests.

The original demonstration functions are preserved for comparison with the expanded implementation.



## 1. How to Run

This notebook contains the original standalone recipe-ranking demonstration and documentation of the expanded SmartPantry AI recipe recommender.

### Original Demonstration

The original demonstration uses sample pantry items and mock recipes to explain ingredient matching, expiration prioritization, and recipe ranking.

These cells can run independently in Google Colab or Jupyter without API keys, external files, or additional Python packages.

### Expanded Functionality

The SmartPantry AI application now includes Spoonacular recipe integration, recall-aware filtering, and verified recipe alternatives.

These features are implemented in `modules/recipe_recommender.py` and validated through automated tests in `tests/test_recipe_recommender.py`.

Testing the current project module requires access to the SmartPantry repository and its dependencies.

Mocked API tests do not require live Spoonacular requests.

### Execution Notes

The original demonstration can be executed independently.

Tests of the expanded implementation should be executed separately using the current project module.

Live API requests are not required for the notebook's automated validation.



## 2. Baseline Recipe Ranking Functions

This section preserves the original recipe-ranking algorithm developed for SmartPantry AI.

The functions demonstrate four core operations:

1. **Ingredient normalization:** Standardizing ingredient names for comparison.
2. **Ingredient matching:** Identifying which recipe ingredients are available in the pantry.
3. **Expiration prioritization:** Assigning higher priority to ingredients approaching expiration.
4. **Recipe scoring:** Combining pantry coverage (60%) and expiration priority (40%) to rank recipes.

### Development Context

These functions were copied from the original `recipe_recommender.py` module to create an independently executable demonstration.

The production implementation has since expanded to include Spoonacular integration, recall-aware filtering, and verified recipe alternatives.

The original functions are retained here to demonstrate the algorithm's development and provide a baseline for comparison.

The expanded implementation is evaluated separately through automated tests.


In [3]:

import os

if not os.path.isdir("/content/smartpantry-ai"):
    !git clone -b docs/phase4-notebook-updates https://github.com/SmartPantry-AI/smartpantry-ai.git /content/smartpantry-ai

%cd /content/smartpantry-ai

%pip install -q -r requirements.txt

print("PASS - SmartPantry environment ready")


Cloning into '/content/smartpantry-ai'...
remote: Enumerating objects: 750, done.
remote: Counting objects: 100% (290/290), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 750 (delta 192), reused 178 (delta 128), pack-reused 460 (from 2)
Receiving objects: 100% (750/750), 16.82 MiB | 41.59 MiB/s, done.
Resolving deltas: 100% (342/342), done.
/content/smartpantry-ai
PASS - SmartPantry environment ready


In [4]:

# SmartPantry AI - Current Recipe Recommender Functions
# Import directly from the production module.

from modules.recipe_recommender import (
    normalize_ingredient_name,
    ingredients_match,
    get_pantry_names,
    get_expiration_points,
    calculate_total_expiration_points,
    check_recipe_ingredients,
    calculate_recipe_expiration_score,
    calculate_pantry_coverage,
    calculate_expiration_coverage,
    calculate_final_score,
    score_recipe,
    rank_recipes,
    convert_spoonacular_recipe,
    rank_spoonacular_recipes,
    fetch_spoonacular_recipes,
    filter_recipes_by_recall_status,
    rank_recall_checked_recipes,
    suggest_verified_recipe_alternatives,
)

print("PASS - Current SmartPantry recipe recommender functions imported")


PASS - Current SmartPantry recipe recommender functions imported



## 3. Sample Pantry and Recipe Candidates

This section uses a controlled dataset to demonstrate how SmartPantry AI evaluates recipe recommendations.

### Sample Pantry

The pantry contains three ingredients:

- **Chicken breast:** 2 pieces, expires in 2 days
- **Spinach:** 1 bag, expires in 1 day
- **Rice:** 2 cups, expires in 30 days

These expiration values allow us to evaluate whether the ranking algorithm prioritizes ingredients that should be used sooner.

### Recipe Candidates

The recipes are predefined examples rather than results retrieved from Spoonacular.

Using controlled sample data makes it possible to examine recipe scores consistently without relying on external API requests.

### Purpose

This experiment evaluates how ingredient availability and expiration priority influence recipe rankings.

The original sample data and ranking functions are preserved to maintain consistency with the baseline demonstration.


In [5]:
pantry = [
    {"name": "chicken breast", "quantity": 2, "unit": "pieces", "days_until_expiration": 2},
    {"name": "spinach", "quantity": 1, "unit": "bag", "days_until_expiration": 1},
    {"name": "rice", "quantity": 2, "unit": "cups", "days_until_expiration": 30},
]
recipes = [
    {"name": "Chicken Rice Bowl", "ingredients": ["chicken breast", "rice"]},
    {"name": "Chicken Spinach Rice Bowl", "ingredients": ["chicken breast", "spinach", "rice", "garlic"]},
]
pantry_weight = 0.60
expiration_weight = 0.40
print("Pantry items:", len(pantry), "| Recipe candidates:", len(recipes))

Pantry items: 3 | Recipe candidates: 2



## 4. Ingredient Normalization and Matching

This section tests how the original SmartPantry AI algorithm compares pantry ingredients with recipe ingredients.

### Normalization

Ingredient names are converted to lowercase and surrounding whitespace is removed to support consistent comparisons.

For example, ` Baby Spinach ` becomes `baby spinach`.

### Ingredient Aliases

The baseline algorithm recognizes selected ingredient variations, including:

- Baby spinach and spinach
- Brown rice and rice
- Chicken breasts and chicken breast

### Matching Validation

The existing assertions check that:

- Supported ingredient aliases match correctly.
- Different ingredients, such as rice and rice vinegar, are not incorrectly matched.
- Ingredient normalization produces the expected result.

### Baseline Limitation

The original matching logic applies aliases to recipe ingredient names but does not handle pantry-side aliases symmetrically.

These tests document the behavior of the original algorithm rather than all capabilities of the expanded production implementation.


In [6]:
assert normalize_ingredient_name(" Baby Spinach ") == "baby spinach"
assert ingredients_match("spinach", "baby spinach") is True
assert ingredients_match("rice", "brown rice") is True
assert ingredients_match("rice", "rice vinegar") is False
print("Normalization and baseline alias tests passed.")
print("Reverse alias currently matches:", ingredients_match("baby spinach", "spinach"))

Normalization and baseline alias tests passed.
Reverse alias currently matches: False



## 5. Recipe Ranking and Scoring

This section evaluates how SmartPantry AI ranks recipe candidates using the baseline scoring algorithm.

### Scoring Formula

The algorithm combines two weighted factors:

- **Pantry Coverage (60%):** Measures the proportion of recipe ingredients already available in the pantry.
- **Expiration Priority (40%):** Rewards recipes that use available ingredients approaching expiration.

**Final Score = (Pantry Coverage × 0.60 + Expiration Coverage × 0.40) × 100**

### Expected Ranking

Using the sample pantry and recipes, the expected results are:

| Rank | Recipe | Score |
|---|---|---|
| 1 | Chicken Spinach Rice Bowl | 85 |
| 2 | Chicken Rice Bowl | 76 |

The Chicken Spinach Rice Bowl ranks higher because it uses spinach, which expires sooner, despite requiring an additional ingredient.

### Purpose

This experiment demonstrates how SmartPantry AI balances ingredient availability with reducing potential food waste.

The scoring results represent the original baseline algorithm, not a live Spoonacular recommendation.


In [7]:
from contextlib import redirect_stdout
from io import StringIO

# Suppress existing debugging messages only in this demonstration cell.
with redirect_stdout(StringIO()):
    ranked = rank_recipes(recipes, pantry, pantry_weight, expiration_weight)
for position, result in enumerate(ranked, 1):
    print(f"{position}. {result['name']}: {result['score']:.1f}")
assert ranked[0]["name"] == "Chicken Spinach Rice Bowl"
assert abs(ranked[0]["score"] - 85.0) < 1e-9
assert abs(ranked[1]["score"] - 76.0) < 1e-9
print("Baseline ranking tests passed.")

1. Chicken Spinach Rice Bowl: 85.0
2. Chicken Rice Bowl: 76.0
Baseline ranking tests passed.



## 6. Individual Recipe Score Analysis

This section examines how the baseline algorithm calculates scores for each recipe.

### Scoring Components

For each recipe, the algorithm evaluates:

- **Available Ingredients:** Ingredients found in the sample pantry.
- **Missing Ingredients:** Ingredients required by the recipe but unavailable in the pantry.
- **Expiration Points:** Points assigned based on how soon available ingredients expire.
- **Pantry Coverage:** The proportion of recipe ingredients available.
- **Expiration Coverage:** The normalized expiration-priority score.
- **Final Score:** The weighted combination of pantry coverage and expiration coverage.

### Purpose

Examining these components helps explain why recipes receive different scores.

In particular, this analysis demonstrates how ingredients approaching expiration can increase a recipe's ranking, supporting SmartPantry AI's goal of reducing potential food waste.

### Evaluation Scope

These calculations use the original sample pantry and predefined recipes. They do not require Spoonacular API requests or access to the Supabase database.


In [8]:
for recipe in recipes:
    with redirect_stdout(StringIO()):
        available, missing, exp_points, exp_coverage, final, pantry_coverage = score_recipe(
            recipe, pantry, pantry_weight, expiration_weight
        )
    print(f"{recipe['name']}: {available} available, {missing} missing, "
          f"pantry coverage {pantry_coverage:.1f}%, expiration coverage {exp_coverage:.1f}%, "
          f"final score {final:.1f}")

Chicken Rice Bowl: 2 available, 0 missing, pantry coverage 100.0%, expiration coverage 40.0%, final score 76.0
Chicken Spinach Rice Bowl: 3 available, 1 missing, pantry coverage 75.0%, expiration coverage 100.0%, final score 85.0




## 7. Known Limitations and Remaining Integration Work

### Original Baseline Limitations

The Phase 3 demonstration used predefined recipes and sample pantry data. It did not directly demonstrate live Spoonacular requests, shared pantry integration, or recall safety filtering.

### Functionality Added Since the Baseline

- **Spoonacular integration:** Recipe fetching, response conversion, and ranking have been implemented.
- **Pantry integration:** Recipe ranking can use inventory retrieved through `PantryManager`.
- **Recall safety:** Recipes can be filtered based on ingredient recall status before ranking.
- **Recipe alternatives:** The recommender can suggest alternatives by removing eligible optional ingredients while blocking recipes with unsafe or unavailable required ingredients.
- **Error handling:** Automated tests cover API errors, unexpected response formats, and incomplete recipe data.

### Remaining Limitations

- Ingredient matching may not recognize every variation or preparation method.
- Live Spoonacular requests depend on API availability and usage limits.
- Incomplete recall verification requires careful handling and does not establish that food is safe.
- The original demonstration does not validate the complete Streamlit and Supabase workflow.
- End-to-end integration with Mochi and the persistent pantry requires separate validation.

### Testing Status

The expanded recipe recommender test suite contains 22 test functions. Their execution results will be documented in Section 8 after validation.

The original demonstration remains preserved as a baseline for comparison.



## 8. Spoonacular Recipe Integration

SmartPantry AI integrates with Spoonacular to retrieve recipe suggestions based on available pantry ingredients.

The current recipe recommender supports three operations:

1. **Recipe Retrieval:** `fetch_spoonacular_recipes()` requests recipes using pantry ingredients.
2. **Recipe Conversion:** `convert_spoonacular_recipe()` converts Spoonacular's response into SmartPantry's recipe format.
3. **Recipe Ranking:** `rank_spoonacular_recipes()` ranks converted recipes using pantry coverage and expiration priority.

### Testing Approach

This demonstration uses a simulated Spoonacular response instead of making a live API request.

The objective is to verify that the current production module can process Spoonacular recipe data and rank the resulting recipes correctly.

Using simulated data makes the experiment repeatable and avoids unnecessary API requests.


In [10]:

# Spoonacular integration test using simulated API data
# No live API request is made.

mock_spoonacular_recipes = [
    {
        "title": "Chicken Rice Bowl",
        "usedIngredients": [
            {"name": "chicken breast"},
            {"name": "rice"}
        ],
        "missedIngredients": []
    },
    {
        "title": "Chicken Spinach Rice Bowl",
        "usedIngredients": [
            {"name": "chicken breast"},
            {"name": "spinach"},
            {"name": "rice"}
        ],
        "missedIngredients": [
            {"name": "garlic"}
        ]
    }
]

# Convert Spoonacular data to SmartPantry's recipe format.
converted_recipes = [
    convert_spoonacular_recipe(recipe)
    for recipe in mock_spoonacular_recipes
]

# Rank recipes using the current production implementation.
ranked_spoonacular = rank_spoonacular_recipes(
    mock_spoonacular_recipes,
    pantry
)

assert len(converted_recipes) == 2
assert converted_recipes[1]["ingredients"] == [
    "chicken breast", "spinach", "rice", "garlic"
]

assert [recipe["name"] for recipe in ranked_spoonacular] == [
    "Chicken Spinach Rice Bowl",
    "Chicken Rice Bowl"
]

assert [recipe["score"] for recipe in ranked_spoonacular] == [85, 76]

print("PASS - Spoonacular recipe conversion")
print("PASS - Spoonacular recipe ranking")

for recipe in ranked_spoonacular:
    print(f'{recipe["name"]}: {recipe["score"]:.0f}')


PASS - Spoonacular recipe conversion
PASS - Spoonacular recipe ranking
Chicken Spinach Rice Bowl: 85
Chicken Rice Bowl: 76



### Historical Test Results

During the original Phase 3 development, the recipe recommender was validated using controlled sample data.

| Test | Result |
|---|---|
| Recipe ranking | PASS |
| Ingredient matching | PASS |
| Expiration prioritization | PASS |
| Empty pantry handling | PASS |
| Empty recipe list handling | PASS |

**Original Recipe Tests:** 5 passed

**Historical Project Regression Tests:** 47 passed

These results represent an earlier development stage and are retained for historical documentation.

### Expanded Automated Test Coverage

The current recipe recommender test suite includes 22 test functions covering the expanded implementation.

| Test Category | Number of Tests | Functionality |
|---|---:|---|
| Recipe Ranking and Ingredient Matching | 5 | Ranking scores, ingredient matching, expiration priority, empty pantry, and empty recipe lists |
| Spoonacular Integration | 7 | Recipe conversion, ranking, API responses, HTTP errors, invalid responses, and missing fields |
| Recall Safety Filtering | 5 | Clear ingredients, recalled ingredients, incomplete verification, missing results, and filtering before ranking |
| Pantry Integration | 1 | Recipe ranking using PantryManager |
| Verified Recipe Alternatives | 3 | Optional ingredient removal and blocking unverified or missing required ingredients |
| Incomplete Verification Acknowledgment | 1 | Acknowledgment handling and blocking potential recall matches |
| **Total** | **22** | |

These automated tests are maintained in `tests/test_recipe_recommender.py`.

The expanded test suite includes mocked external requests, allowing API-related behavior to be tested without making live Spoonacular requests.


### Automated Test Execution Results

**Execution Environment:** Google Colab  
**Python Version:** 3.13.18  
**Test Framework:** pytest 8.4.2  
**Test File:** `tests/test_recipe_recommender.py`

| Result | Count |
|---|---:|
| Tests Collected | 22 |
| Tests Passed | 22 |
| Tests Failed | 0 |
| Success Rate | 100% |
| Execution Time | 0.06 seconds |

**Final Status: PASS**

All 22 automated recipe recommender tests passed successfully.

The tests verified recipe ranking, ingredient matching, expiration prioritization, Spoonacular integration, API error handling, recall safety filtering, pantry integration, and verified recipe alternatives.

All tests used controlled data or mocked external requests where applicable. No live Spoonacular API calls were required.

### Current Notebook Validation Results

The updated notebook was executed in Google Colab using functions imported directly from the current SmartPantry AI recipe recommender module.

| Validation | Result |
|---|---|
| Production module imports | PASS |
| Baseline ingredient matching | PASS |
| Baseline recipe ranking | PASS |
| Individual recipe score analysis | PASS |
| Spoonacular conversion and ranking | PASS |
| Recall safety: all ingredients clear | PASS |
| Recall safety: recalled ingredient | PASS |
| Recall safety: incomplete verification | PASS |
| Alternative: optional ingredient removal | PASS |
| Alternative: unverified required ingredient | PASS |
| Alternative: missing required ingredient | PASS |

### Results Summary

All executed notebook validation scenarios passed.

The tests confirmed that the current recipe recommender correctly processes simulated Spoonacular responses, ranks recipes using pantry availability and expiration priority, filters recipes based on recall verification, and handles optional and required ingredients when generating alternatives.

The original recipe ranking results remained consistent:

- Chicken Spinach Rice Bowl: 85
- Chicken Rice Bowl: 76

### Automated Test Suite

The repository also contains 22 dedicated recipe recommender test functions in `tests/test_recipe_recommender.py`.

These tests provide additional coverage of API errors, invalid responses, empty inputs, and integration behavior.

The repository test suite has not yet been executed as part of this notebook update.

### Conclusion

The updated notebook demonstrates the current SmartPantry AI recipe recommendation functionality using the production module rather than outdated copied functions.

All notebook validation scenarios completed successfully without live Spoonacular API requests.

The results support the correctness of the tested behaviors, while broader integration and live API reliability require separate validation.

In [9]:

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_recipe_recommender.py",
        "-v"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise AssertionError("Recipe recommender tests failed")

print("PASS - Recipe recommender test suite completed")


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/smartpantry-ai
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 22 items

tests/test_recipe_recommender.py::test_recipe_ranking PASSED             [  4%]
tests/test_recipe_recommender.py::test_ingredient_matching PASSED        [  9%]
tests/test_recipe_recommender.py::test_expiration_priority PASSED        [ 13%]
tests/test_recipe_recommender.py::test_empty_pantry PASSED               [ 18%]
tests/test_recipe_recommender.py::test_empty_recipe_list PASSED          [ 22%]
tests/test_recipe_recommender.py::test_spoonacular_conversion PASSED     [ 27%]
tests/test_recipe_recommender.py::test_spoonacular_ranking_integration PASSED [ 31%]
tests/test_recipe_recommender.py::test_rank_spoonacular_recipes PASSED   [ 36%]
tests/test_recipe_reco